In [1]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

from caveat.encoding.continuous import ContinuousEncoder
from caveat.label_encoding import TokenAttributeEncoder
from caveat.mine_xy import DataModule, MutualInformationEstimator
from caveat.models.continuous.cvae_lstm import (
    ConcatEncoder,
    HiddenLabel,
    LabelEncoder,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/torchmetrics/__init__.py:31: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.4)
  import scipy.signal


Device: cuda


/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


In [2]:
def latest(path: Path):
    versions = sorted(
        [
            d
            for d in path.iterdir()
            if d.is_dir() and d.name.startswith("version")
        ]
    )
    return Path(versions[-1])


def iter_models(path: Path):
    for dir in path.iterdir():
        if dir.is_dir():
            yield latest(dir)

In [6]:
label_encoders = {
    # "gender": TokenAttributeEncoder(config={"gender": "nominal"}),
    # "age_group": TokenAttributeEncoder(config={"age_group": "nominal"}),
    # "car_access": TokenAttributeEncoder(config={"car_access": "nominal"}),
    # "work_status": TokenAttributeEncoder(config={"work_status": "nominal"}),
    # "income": TokenAttributeEncoder(config={"income": "nominal"}),
    "all": TokenAttributeEncoder(
        config={
            "age": "nominal",
            "sex": "nominal",
            "employment": "nominal",
            "hh_income": "nominal",
            "hh_zone": "nominal",
            "day": "nominal",
            "vehicles": "nominal",
            "access_egress_distance": "nominal",
        }
    )
}
schedule_encoder = ContinuousEncoder()


def custom_loader(
    label_encoder,
    schedule_encoder,
    n: int = 5,
    shuffle_y: bool = False,
    zero_y: bool = False,
):
    ys = pd.read_csv(Path("~/Data/foundata/out/nts/2023/attributes_binned.csv"))
    ys, _ = label_encoder.encode(ys)
    if shuffle_y:
        ys = ys[torch.randperm(ys.shape[0])]
    if zero_y:
        ys = ys * 0
    xs = pd.read_csv(Path("~/Data/foundata/out/nts/2023/activities.csv"))
    if "duration" not in xs.columns:
        xs["duration"] = xs.end - xs.start
    xs = schedule_encoder.encode(schedules=xs, labels=ys, label_weights=None)
    return xs

Label Encoder TokenAttributeEncoder initialised with:
            Label weighting: unit
            Joint weighting: unit
            
Continuous Encoder initialised with:
        max_length: 12
        norm_duration: 1440
        jitter: 0
        fix_durations: stretch
        (act) weighting: unit
        (seq) joint weighting: unit
        trim eos: True
        


In [7]:
class MinerNet(nn.Module):
    def __init__(
        self,
        encodings,
        max_length,
        label_embed_sizes,
        hidden_size=256,
        encoder_depth=2,
        block_depth=2,
        dropout=0.2,
    ):
        super(MinerNet, self).__init__()

        self.label_embed = LabelEncoder(
            label_embed_sizes=label_embed_sizes, hidden_size=hidden_size
        )

        self.hidden = HiddenLabel(
            hidden_size=hidden_size,
            hidden_layers=encoder_depth,
            labels_size=hidden_size,
            dropout=dropout,
            activation=False,
        )

        self.schedule_encoder = ConcatEncoder(
            input_size=encodings,
            hidden_size=hidden_size,
            hidden_layers=encoder_depth,
            labels_size=hidden_size,
            max_length=max_length,
            dropout=dropout,
        )

        size = encoder_depth * hidden_size * 2

        blocks = []
        for _ in range(block_depth - 1):
            blocks.append(nn.Linear(size, hidden_size))
            if dropout > 0:
                blocks.append(nn.Dropout(dropout))
            blocks.append(nn.LeakyReLU())
            size = hidden_size
        self.blocks = nn.Sequential(*blocks, nn.Linear(size, 1))

    def forward(self, xs, ys):
        h0 = self.label_embed(ys.long())
        h1 = self.hidden(h0)
        h2 = self.schedule_encoder(xs, h0, h1)
        return self.blocks(h2)

In [8]:
hypers = {}
for alpha in [1]:
    results = {}
    for name, label_encoder in label_encoders.items():
        model_results = []

        for i in range(5):
            dataset = custom_loader(label_encoder, schedule_encoder)
            logger = TensorBoardLogger("logs/xy", name=f"{name}_{alpha}_{i}")
            loader = DataModule(
                dataset=dataset,
                val_split=0.1,
                test_split=0.1,
                batch_size=1024,
                num_workers=8,
                pin_memory=False,
            )

            net = MinerNet(
                encodings=dataset.activity_encodings,
                max_length=schedule_encoder.max_length,
                label_embed_sizes=label_encoder.label_kwargs[
                    "label_embed_sizes"
                ],
                hidden_size=256,
                encoder_depth=2,
                block_depth=2,
                dropout=0.2,
            )

            kwargs = {"alpha": alpha, "lr": 1e-3, "weight_decay": 1e-3}
            model = MutualInformationEstimator(net=net, **kwargs)
            trainer = Trainer(
                min_epochs=10,
                max_epochs=500,
                accelerator=device,
                devices=1,
                enable_progress_bar=False,
                logger=logger,
                enable_checkpointing=True,
                callbacks=[
                    EarlyStopping(monitor="val_loss", patience=50),
                    ModelCheckpoint(
                        monitor="val_loss",
                        save_top_k=2,
                        save_weights_only=False,
                    ),
                ],
            )
            trainer.fit(model, datamodule=loader)
            mi = trainer.test(ckpt_path="best", datamodule=loader)[0]["test_mi"]
            model_results.append(mi)
        results[name] = {
            "mean": np.mean(model_results),
            "var": np.var(model_results),
        }
    hypers[alpha] = results

/tmp/ipykernel_3222401/203866817.py:30: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  ys = pd.read_csv(Path("~/Data/foundata/out/nts/2023/attributes_binned.csv"))
/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net         │ MinerNet │  1.6 M │ train │     0 │
│ 1 │ energy_loss │ Mine     │  1.6 M │ train │     0 │
└───┴─────────────┴──────────┴────────┴───────┴───────┘

Trainable params: 1.6 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.6 M                                                                                                
Total estimated model params size (MB): 6                                                                          
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:317: The number of training batches (45) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    -0.5898963809013367    │
│          test_mi          │    0.5898963809013367     │
└───────────────────────────┴───────────────────────────┘

...

/tmp/ipykernel_3222401/203866817.py:30: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  ys = pd.read_csv(Path("~/Data/foundata/out/nts/2023/attributes_binned.csv"))


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net         │ MinerNet │  1.6 M │ train │     0 │
│ 1 │ energy_loss │ Mine     │  1.6 M │ train │     0 │
└───┴─────────────┴──────────┴────────┴───────┴───────┘

Trainable params: 1.6 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.6 M                                                                                                
Total estimated model params size (MB): 6                                                                          
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:317: The number of training batches (45) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    -0.5035991668701172    │
│          test_mi          │    0.5035991668701172     │
└───────────────────────────┴───────────────────────────┘

In [9]:
for depth, res in hypers.items():
    print(f"Depth {depth}:")
    for name, result in res.items():
        print(f"\tResults for {name}: {result}")

Depth 1:
	Results for all: {'mean': np.float64(0.5278775453567505), 'var': np.float64(0.0017048238499322108)}


In [7]:
cols = [
    "ethnicity",
    "education",
    "license",
    "work_status",
    "mobility",
    "wheelchair_user",
    "WfH",
    "ticket_holder",
    "health",
    "area",
    "hh_size",
    "hh_composition",
    "hh_children",
    "hh_cars",
    "hh_bikes",
    "hh_motorcycles",
    "blue_badge",
    "num_license_holders",
]
label_encoders = {n: TokenAttributeEncoder(config={n: "nominal"}) for n in cols}

for alpha in [1]:
    results = {}
    for name, label_encoder in label_encoders.items():
        model_results = []

        for i in range(5):
            dataset = custom_loader(label_encoder, schedule_encoder)
            logger = TensorBoardLogger("logs/xy", name=f"{name}_{alpha}_{i}")
            loader = DataModule(
                dataset=dataset,
                val_split=0.1,
                test_split=0.1,
                batch_size=1024,
                num_workers=8,
                pin_memory=False,
            )

            net = MinerNet(
                encodings=dataset.activity_encodings,
                max_length=schedule_encoder.max_length,
                label_embed_sizes=label_encoder.label_kwargs[
                    "label_embed_sizes"
                ],
                hidden_size=256,
                encoder_depth=2,
                block_depth=2,
                dropout=0.2,
            )

            kwargs = {"alpha": alpha, "lr": 1e-3, "weight_decay": 1e-3}
            model = MutualInformationEstimator(net=net, **kwargs)
            trainer = Trainer(
                min_epochs=10,
                max_epochs=500,
                accelerator=device,
                devices=1,
                enable_progress_bar=False,
                logger=logger,
                enable_checkpointing=True,
                callbacks=[
                    EarlyStopping(monitor="val_loss", patience=50),
                    ModelCheckpoint(
                        monitor="val_loss",
                        save_top_k=2,
                        save_weights_only=False,
                    ),
                ],
            )
            trainer.fit(model, datamodule=loader)
            mi = trainer.test(ckpt_path="best", datamodule=loader)[0]["test_mi"]
            model_results.append(mi)
        results[name] = {
            "mean": np.mean(model_results),
            "var": np.var(model_results),
        }
    hypers[alpha] = results

Label Encoder TokenAttributeEncoder initialised with:
            Label weighting: unit
            Joint weighting: unit
            
Label Encoder TokenAttributeEncoder initialised with:
            Label weighting: unit
            Joint weighting: unit
            
Label Encoder TokenAttributeEncoder initialised with:
            Label weighting: unit
            Joint weighting: unit
            
Label Encoder TokenAttributeEncoder initialised with:
            Label weighting: unit
            Joint weighting: unit
            
Label Encoder TokenAttributeEncoder initialised with:
            Label weighting: unit
            Joint weighting: unit
            
Label Encoder TokenAttributeEncoder initialised with:
            Label weighting: unit
            Joint weighting: unit
            
Label Encoder TokenAttributeEncoder initialised with:
            Label weighting: unit
            Joint weighting: unit
            
Label Encoder TokenAttributeEncoder initialised with:
 

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (47) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │   -0.022114671766757965   │
│          test_mi          │   0.022114671766757965    │
└───────────────────────────┴───────────────────────────┘

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (47) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │   -0.012843221426010132   │
│          test_mi          │   0.012843221426010132    │
└───────────────────────────┴───────────────────────────┘

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │   -0.025030985474586487   │
│          test_mi          │   0.025030985474586487    │
└───────────────────────────┴───────────────────────────┘

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (47) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │   -0.021182354539632797   │
│          test_mi          │   0.021182354539632797    │
└───────────────────────────┴───────────────────────────┘

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (47) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │   -0.018584787845611572   │
│          test_mi          │   0.018584787845611572    │
└───────────────────────────┴───────────────────────────┘

In [8]:
for depth, res in hypers.items():
    print(f"Depth {depth}:")
    for name, result in res.items():
        print(f"\tResults for {name}: {result}")

Depth 1:
	Results for ethnicity: {'mean': 0.019015155173838137, 'var': 3.189956023459589e-05}
	Results for education: {'mean': 0.17116478085517883, 'var': 0.00010085279770226663}
	Results for license: {'mean': 0.19030461311340333, 'var': 0.00041194683329527133}
	Results for work_status: {'mean': 0.2233883947134018, 'var': 5.551786688050342e-05}
	Results for mobility: {'mean': 0.18208744525909423, 'var': 0.0001064664158449169}
	Results for wheelchair_user: {'mean': 2.0397139550709652e-07, 'var': 1.9249004243775993e-14}
	Results for WfH: {'mean': 0.11585043370723724, 'var': 0.00016845196755659319}
	Results for ticket_holder: {'mean': 0.07306414544582367, 'var': 3.1845449530156776e-05}
	Results for health: {'mean': 0.19089796543121337, 'var': 7.256539384089321e-05}
	Results for area: {'mean': 0.00937522342428565, 'var': 5.763046588764913e-06}
	Results for hh_size: {'mean': 0.11652848273515701, 'var': 0.00011455876740272596}
	Results for hh_composition: {'mean': 0.15147133767604828, 'var':